# Semana 5: SQL DML y JOINs
## Notebook Conceptual (NB1) – JOINs con sets en Python

### Propósito de la sesión
Dominar las operaciones de combinación de tablas (JOINs) implementándolas manualmente en Python para entender su funcionamiento interno, y luego comparar con la implementación optimizada de `pandas.merge()`. Esta comprensión es fundamental para escribir consultas SQL eficientes y para entender cómo los motores de bases de datos combinan datos.

### Objetivos de aprendizaje
*   Representar tablas como listas de diccionarios en Python.
*   Implementar manualmente los diferentes tipos de JOIN: **INNER**, **LEFT**, **RIGHT**, **FULL OUTER**.
*   Entender la lógica de combinación basada en claves.
*   Comparar nuestras implementaciones con `pandas.merge()`.
*   Reflexionar sobre la complejidad algorítmica de los JOINs.

## Configuración Inicial

Importamos las librerías necesarias y definimos datos de ejemplo.

In [ ]:
# Importamos pandas para comparación
import pandas as pd
from IPython.display import display

print("🎯 Librerías importadas correctamente.")

---
## 1. Creación de Datos de Ejemplo

Simularemos dos tablas: **clientes** y **pedidos**. Un cliente puede tener cero, uno o varios pedidos.

In [ ]:
# Tabla de clientes
clientes = [
    {'id': 1, 'nombre': 'Juan Pérez'},
    {'id': 2, 'nombre': 'María García'},
    {'id': 3, 'nombre': 'Carlos López'},
    {'id': 4, 'nombre': 'Ana Martínez'}
]

# Tabla de pedidos
pedidos = [
    {'id_pedido': 101, 'cliente_id': 1, 'producto': 'Laptop', 'cantidad': 1},
    {'id_pedido': 102, 'cliente_id': 2, 'producto': 'Mouse', 'cantidad': 2},
    {'id_pedido': 103, 'cliente_id': 1, 'producto': 'Monitor', 'cantidad': 1},
    {'id_pedido': 104, 'cliente_id': 3, 'producto': 'Teclado', 'cantidad': 1},
    {'id_pedido': 105, 'cliente_id': 5, 'producto': 'Auriculares', 'cantidad': 1}  # cliente_id 5 no existe en clientes
]

# Convertimos a DataFrames de pandas para referencia
df_clientes = pd.DataFrame(clientes)
df_pedidos = pd.DataFrame(pedidos)

print("🔷 Tabla Clientes:")
display(df_clientes)

print("\n🔶 Tabla Pedidos:")
display(df_pedidos)

---
## 2. Función Auxiliar para Mostrar Resultados

Definimos una función para imprimir los resultados de nuestros JOINs manuales de forma legible.

In [ ]:
def mostrar_resultado(resultado, titulo):
    """Muestra el resultado de un JOIN como DataFrame."""
    if not resultado:
        print(f"{titulo}: (vacío)")
    else:
        df = pd.DataFrame(resultado)
        print(f"\n{titulo}:")
        display(df)

---
## 3. Implementación Manual de JOINs

### 3.1. INNER JOIN

El INNER JOIN devuelve solo las filas que tienen correspondencia en ambas tablas según la clave de unión.

In [ ]:
def inner_join(left, right, left_key, right_key):
    """
    Implementación manual de INNER JOIN.
    left: lista de diccionarios (tabla izquierda)
    right: lista de diccionarios (tabla derecha)
    left_key: clave de unión en la tabla izquierda
    right_key: clave de unión en la tabla derecha
    """
    resultado = []
    for fila_left in left:
        for fila_right in right:
            if fila_left[left_key] == fila_right[right_key]:
                # Combinamos ambas filas
                nueva_fila = fila_left.copy()
                nueva_fila.update(fila_right)
                resultado.append(nueva_fila)
    return resultado

# Probamos INNER JOIN
inner_result = inner_join(clientes, pedidos, 'id', 'cliente_id')
mostrar_resultado(inner_result, "INNER JOIN manual")

### 3.2. LEFT JOIN (LEFT OUTER JOIN)

LEFT JOIN devuelve todas las filas de la tabla izquierda, y las coincidentes de la derecha. Si no hay coincidencia, las columnas de la derecha son `None` (o `NULL` en SQL).

In [ ]:
def left_join(left, right, left_key, right_key):
    """
    Implementación manual de LEFT JOIN.
    """
    resultado = []
    for fila_left in left:
        # Buscamos coincidencias en right
        encontradas = False
        for fila_right in right:
            if fila_left[left_key] == fila_right[right_key]:
                nueva_fila = fila_left.copy()
                nueva_fila.update(fila_right)
                resultado.append(nueva_fila)
                encontradas = True
        if not encontradas:
            # Si no hay coincidencia, añadimos la fila izquierda con valores None en las columnas de right
            nueva_fila = fila_left.copy()
            # Añadimos las claves de right con None (tomamos las claves del primer elemento de right si existe)
            if right:
                for k in right[0].keys():
                    if k not in nueva_fila:
                        nueva_fila[k] = None
            resultado.append(nueva_fila)
    return resultado

left_result = left_join(clientes, pedidos, 'id', 'cliente_id')
mostrar_resultado(left_result, "LEFT JOIN manual")

### 3.3. RIGHT JOIN (RIGHT OUTER JOIN)

RIGHT JOIN es análogo al LEFT, pero con la tabla derecha como maestra. Podemos implementarlo como LEFT JOIN intercambiando las tablas, o manualmente.

In [ ]:
def right_join(left, right, left_key, right_key):
    """
    Implementación manual de RIGHT JOIN (usando left como tabla derecha).
    """
    resultado = []
    for fila_right in right:
        encontradas = False
        for fila_left in left:
            if fila_right[right_key] == fila_left[left_key]:
                nueva_fila = fila_left.copy()
                nueva_fila.update(fila_right)
                resultado.append(nueva_fila)
                encontradas = True
        if not encontradas:
            nueva_fila = {}
            # Copiamos todas las columnas de left con None
            if left:
                for k in left[0].keys():
                    nueva_fila[k] = None
            # Añadimos la fila de right
            nueva_fila.update(fila_right)
            resultado.append(nueva_fila)
    return resultado

right_result = right_join(clientes, pedidos, 'id', 'cliente_id')
mostrar_resultado(right_result, "RIGHT JOIN manual")

### 3.4. FULL OUTER JOIN

FULL OUTER JOIN combina LEFT y RIGHT: devuelve todas las filas de ambas tablas, rellenando con `None` donde no hay coincidencia.

In [ ]:
def full_outer_join(left, right, left_key, right_key):
    """
    Implementación manual de FULL OUTER JOIN.
    """
    resultado = []
    # Primero, hacemos un LEFT JOIN
    for fila_left in left:
        encontradas = False
        for fila_right in right:
            if fila_left[left_key] == fila_right[right_key]:
                nueva_fila = fila_left.copy()
                nueva_fila.update(fila_right)
                resultado.append(nueva_fila)
                encontradas = True
        if not encontradas:
            nueva_fila = fila_left.copy()
            if right:
                for k in right[0].keys():
                    if k not in nueva_fila:
                        nueva_fila[k] = None
            resultado.append(nueva_fila)
    
    # Luego, añadimos las filas de right que no fueron incluidas (RIGHT part)
    # Para eso, necesitamos saber qué claves de right ya aparecieron
    claves_right_ya_incluidas = set()
    for fila in resultado:
        if right_key in fila and fila[right_key] is not None:
            claves_right_ya_incluidas.add(fila[right_key])
    
    for fila_right in right:
        if fila_right[right_key] not in claves_right_ya_incluidas:
            nueva_fila = {}
            if left:
                for k in left[0].keys():
                    nueva_fila[k] = None
            nueva_fila.update(fila_right)
            resultado.append(nueva_fila)
    
    return resultado

full_result = full_outer_join(clientes, pedidos, 'id', 'cliente_id')
mostrar_resultado(full_result, "FULL OUTER JOIN manual")

---
## 4. Comparación con `pandas.merge()`

Ahora usamos la función `merge` de pandas para realizar los mismos JOINs y comparamos resultados.

In [ ]:
# INNER JOIN con pandas
inner_pd = pd.merge(df_clientes, df_pedidos, left_on='id', right_on='cliente_id', how='inner')
mostrar_resultado(inner_pd.to_dict('records'), "INNER JOIN con pandas")

# LEFT JOIN con pandas
left_pd = pd.merge(df_clientes, df_pedidos, left_on='id', right_on='cliente_id', how='left')
mostrar_resultado(left_pd.to_dict('records'), "LEFT JOIN con pandas")

# RIGHT JOIN con pandas
right_pd = pd.merge(df_clientes, df_pedidos, left_on='id', right_on='cliente_id', how='right')
mostrar_resultado(right_pd.to_dict('records'), "RIGHT JOIN con pandas")

# FULL OUTER JOIN con pandas
full_pd = pd.merge(df_clientes, df_pedidos, left_on='id', right_on='cliente_id', how='outer')
mostrar_resultado(full_pd.to_dict('records'), "FULL OUTER JOIN con pandas")

---
## 5. Verificación de Resultados

Comprobemos que nuestros JOINs manuales producen los mismos resultados que pandas (ignorando el orden de las filas y posibles diferencias en claves).

In [ ]:
def normalizar_resultado(lista_dict):
    """Convierte lista de dicts a tuplas ordenadas para comparación."""
    return sorted([tuple(sorted(d.items())) for d in lista_dict])

inner_manual_norm = normalizar_resultado(inner_result)
inner_pd_norm = normalizar_resultado(inner_pd.to_dict('records'))

print("INNER JOIN iguales?", inner_manual_norm == inner_pd_norm)

left_manual_norm = normalizar_resultado(left_result)
left_pd_norm = normalizar_resultado(left_pd.to_dict('records'))
print("LEFT JOIN iguales?", left_manual_norm == left_pd_norm)

right_manual_norm = normalizar_resultado(right_result)
right_pd_norm = normalizar_resultado(right_pd.to_dict('records'))
print("RIGHT JOIN iguales?", right_manual_norm == right_pd_norm)

full_manual_norm = normalizar_resultado(full_result)
full_pd_norm = normalizar_resultado(full_pd.to_dict('records'))
print("FULL OUTER JOIN iguales?", full_manual_norm == full_pd_norm)

---
## 6. Discusión sobre Complejidad y Eficiencia

*   Nuestra implementación manual usa bucles anidados, lo que tiene complejidad **O(n * m)** en el peor caso. Para tablas grandes, esto es ineficiente.
*   Los motores de bases de datos utilizan algoritmos optimizados como:
    *   **Nested Loop**: similar a nuestro bucle, pero con índices puede ser más rápido.
    *   **Hash Join**: construye un índice hash en memoria para una de las tablas, reduciendo a O(n + m) en promedio.
    *   **Merge Join**: ordena las tablas y luego las fusiona, útil si ya están ordenadas.
*   pandas también usa implementaciones optimizadas en C.

**Reflexión**: Nuestra implementación manual es educativa, pero en la práctica confiamos en motores optimizados.

---
## Ejercicios para el Estudiante

1.  **Añadir más datos:** Extiende las tablas con más filas (ej. 10 clientes, 20 pedidos) y mide el tiempo de ejecución de tus JOINs manuales. Compara con `pandas.merge()`.

2.  **JOIN con múltiples condiciones:** Modifica la función `inner_join` para que acepte una lista de condiciones (ej. igualdad en varias claves). Pruébala con una clave compuesta.

3.  **Self-Join:** Implementa un self-join manual. Por ejemplo, usando la tabla de empleados con jefe_id (puedes crear una lista de empleados). Encuentra el nombre del empleado y el de su jefe.

4.  **Optimización con diccionarios:** Reescribe `inner_join` para que primero construya un diccionario con los elementos de la tabla derecha indexados por clave, reduciendo la complejidad a O(n + m). Mide la mejora.

5.  **Reflexión:** ¿Por qué los motores de bases de datos no suelen usar bucles anidados para tablas grandes? ¿Qué estrategias emplean?

---
## Conclusiones de la Sesión

*   Implementamos manualmente los principales tipos de **JOIN** (INNER, LEFT, RIGHT, FULL) usando bucles en Python, comprendiendo su lógica interna.
*   Comparamos nuestras implementaciones con `pandas.merge()`, verificando que producen los mismos resultados.
*   Entendimos que los JOINs son operaciones fundamentales para combinar datos de tablas relacionadas.
*   Reflexionamos sobre la complejidad algorítmica y las optimizaciones que realizan los motores de bases de datos.
*   Esta comprensión es esencial para escribir consultas SQL eficientes y para entender cómo los SGBD procesan los JOINs.

¡Ahora sabes lo que ocurre "bajo el capó" cuando ejecutas un JOIN en SQL!